# G03-B — Frozen N01 Lesion-Aware Ensemble Replay

**Kaggle notebook:** `g03b-lesion-aware-all-ensembles-s42`  
**Required inputs:** completed `g02-b-v2-lesion-aware-n01-ham-bcn` output with `reports/g02b_status.json` and all prediction chunks; and the completed `lesion-r07-n01-all-ensembles-s42` output with `reports/final_ensemble_config.json`, **all six fitted `models/Stack_*.joblib`**, `predictions/*_meta_val.npz`, and `reports/final_status.json`.  
**Compute:** CPU, Internet OFF. No image downloads; no fitting.  
**Replays all R07 candidate specifications** (205 for the uploaded real R07 run) without retuning weights, gates, logistic regression, or class bias. Checks provenance and aligns by image IDs.  
**Statistical scope:** G02-B's 2,109 HAM/BCN **N01 validation images**, which were also used to select and bias-adjust R07. The source-specific tables are **descriptive, not independent generalization estimates**. The locked N01 test is not touched.  
**Class handling:** eight-output evaluation primary; restricted closed-set analysis secondary. Never silently drop unavailable classes.

## 1. Strict input loading and reusable evaluation functions

In [1]:
import os,json,hashlib,time,math
from pathlib import Path
from collections import OrderedDict
import numpy as np,pandas as pd,joblib
from scipy.special import softmax
from sklearn.metrics import f1_score,accuracy_score,log_loss

SEED=42
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path(os.getenv('G03_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').is_dir() else '/mnt/data')).resolve()
WORK=Path(os.getenv('G03_WORK_DIR','/kaggle/working' if Path('/kaggle/working').is_dir() else '/mnt/data')).resolve()

def req(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def jwrite(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str));tmp.replace(path)
def get_json(p):return json.loads(Path(p).read_text())
def find_status(filename,stage):
    hits=[]
    for p in INPUT.rglob(filename):
        if not p.is_file():continue
        j=get_json(p)
        if j.get('stage')==stage and j.get('status')=='COMPLETED':hits.append((p,j))
    req(len(hits)==1,f'Expected one completed {stage} {filename}; found {len(hits)}. Attach its Kaggle notebook output. Paths: {[str(a[0]) for a in hits[:6]]}')
    return hits[0]
def normalized(a):
    a=np.asarray(a,dtype=np.float64)
    req(a.ndim==2 and a.shape[1]==8 and np.isfinite(a).all() and (a>=0).all(),'Invalid [N,8] predictions')
    a=np.maximum(a,1e-12)
    return a/a.sum(1,keepdims=True)
def tscale(p,t):
    req(np.isfinite(t) and float(t)>0,'Invalid temperature')
    return softmax(np.log(np.clip(p,1e-12,1))/float(t),axis=1)
def bias(p,b):
    b=np.asarray(b,dtype=float)
    req(b.shape==(8,) and np.isfinite(b).all(),'Invalid class-bias vector')
    return softmax(np.log(np.clip(p,1e-12,1))+b[None,:],axis=1)
def combine(d,keys,weights=None,kind='arithmetic'):
    P=np.stack([d[s] for s in keys],axis=0)
    w=np.ones(len(keys),dtype=float)/len(keys) if weights is None else np.asarray(weights,dtype=float)
    req(w.shape==(len(keys),) and np.isfinite(w).all() and (w>=0).all() and np.isclose(w.sum(),1,atol=2e-4),'Invalid ensemble weights')
    w=w/w.sum()
    if kind=='arithmetic':return normalized(np.einsum('snc,s->nc',P,w,optimize=True))
    if kind=='geometric':return softmax(np.einsum('snc,s->nc',np.log(np.clip(P,1e-12,1)),w,optimize=True),axis=1)
    if kind=='vote':
        onehot=np.eye(8)[P.argmax(2)]
        return normalized(np.einsum('snc,s->nc',onehot,w,optimize=True)+.0001*np.einsum('snc,s->nc',P,w,optimize=True))
    raise RuntimeError(f'Unknown combination: {kind}')
def gated(d,keys,measure,beta,prior=None):
    p=np.stack([d[s] for s in keys],axis=1)
    if measure=='confidence':q=p.max(2)
    elif measure=='entropy':q=1.+(p*np.log(np.clip(p,1e-12,1))).sum(2)/math.log(8)
    elif measure=='margin':
        top=np.partition(p,-2,axis=2)[:,:,-2:];q=top.max(2)-top.min(2)
    else:raise RuntimeError(f'Unknown gate measure {measure}')
    prior_term=0 if prior is None else np.log(np.asarray(prior,dtype=float)[None,:])
    w=softmax(float(beta)*q+prior_term,axis=1)
    return normalized(np.einsum('nsc,ns->nc',p,w,optimize=True))
def entropy(p):return -np.sum(p*np.log(np.clip(p,1e-12,1)),axis=1)
def stats_from_prob(p):
    t=np.partition(p,-2,axis=1)[:,-2:]
    return np.stack([p.max(1),t.max(1)-t.min(1),entropy(p)/math.log(8)],axis=1)
def stacked_features(d,keys,mode):
    x=np.concatenate([d[s] if mode=='raw' else np.log(np.clip(d[s],1e-12,1)) for s in keys],axis=1)
    if mode=='extended':x=np.concatenate([x]+[stats_from_prob(d[s]) for s in keys],axis=1)
    return x
def old_thesis_features(d,keys):
    parts=[]
    for s in keys:
        p=d[s];parts.extend([p,np.log(np.clip(p,1e-12,1)),p.max(1,keepdims=True),entropy(p)[:,None]])
    return np.concatenate(parts,axis=1).astype(np.float32)
def aligned_predict(model,x):
    clf=model.named_steps.get('clf',model.named_steps.get('classifier',None)) if hasattr(model,'named_steps') else model
    req(clf is not None and hasattr(clf,'classes_'),'Fitted stacking model missing classes_')
    cls=np.asarray(clf.classes_).astype(int)
    req(np.array_equal(cls,np.arange(8)),f'Unexpected trained stacker classes: {cls}')
    p=model.predict_proba(x)
    return normalized(p)
def ece(y,p,bins=15):
    pred=p.argmax(1);conf=p.max(1);edges=np.linspace(0,1,bins+1)
    out=0.
    for i in range(bins):
        mask=(conf>=edges[i]) & (conf<(edges[i+1] if i<bins-1 else 1.00000001))
        if mask.any():out+=mask.mean()*abs((pred[mask]==y[mask]).mean()-conf[mask].mean())
    return float(out)
def subset_score(y,p):
    y=np.asarray(y,dtype=int);p=normalized(p);pred=p.argmax(1)
    seen=np.unique(y)
    req(len(seen)>=2,'At least two represented diagnoses required')
    recalls=[(pred[y==c]==c).mean() for c in seen]
    missing=~np.isin(pred,seen)
    return dict(n=len(y),classes_present=';'.join(CLASSES[c] for c in seen),n_present=len(seen),
        balanced_accuracy=float(np.mean(recalls)),accuracy=float((pred==y).mean()),
        macro_f1=float(f1_score(y,pred,labels=seen,average='macro',zero_division=0)),
        macro_f1_eight=float(f1_score(y,pred,labels=np.arange(8),average='macro',zero_division=0)),
        nll_eight=float(log_loss(y,p,labels=np.arange(8))),ece_15=ece(y,p),
        unsupported_prediction_rate=float(missing.mean()),unsupported_predictions=int(missing.sum()))
def restricted(y,p):
    present=np.unique(np.asarray(y,dtype=int))
    a=np.asarray(p[:,present],dtype=float)
    z=np.zeros_like(p,dtype=float)
    z[:,present]=a/np.clip(a.sum(1,keepdims=True),1e-12,None)
    return z

def group_bootstrap(y,p,groups,B=300,seed=SEED):
    y=np.asarray(y,dtype=int);p=normalized(p);groups=np.asarray(groups,dtype=str)
    req(len(y)==len(p)==len(groups),'Bootstrap inputs not aligned')
    groups=np.asarray([f'unknown-image-{i}' if str(g).strip()=='' or str(g).upper().endswith(':UNKNOWN') else str(g) for i,g in enumerate(groups)])
    uniq,inv=np.unique(groups,return_inverse=True)
    indices=[np.flatnonzero(inv==i) for i in range(len(uniq))]
    rng=np.random.default_rng(seed);seen=set(y);values=[]
    for _ in range(B):
        take=np.concatenate([indices[i] for i in rng.integers(0,len(indices),len(indices))])
        yy=y[take]
        if set(yy)!=seen:continue
        pred=p[take].argmax(1)
        values.append(np.mean([(pred[yy==c]==c).mean() for c in seen]))
    return {'low':float(np.percentile(values,2.5)) if len(values)>=max(30,B//5) else None,
            'high':float(np.percentile(values,97.5)) if len(values)>=max(30,B//5) else None,
            'valid_resamples':len(values),'groups':len(uniq)}

def load_chunks(status_path,status,sources,stage):
    root=status_path.parent.parent
    chunks=sorted(status['chunks'],key=lambda x:x['start'])
    req(chunks and chunks[0]['start']==0 and chunks[-1]['end']==status['images'],'Incomplete chunk range')
    ids=[];ys=[];origin=[];group=[];data={s:[] for s in sources}
    offset=0
    for row in chunks:
        req(row['start']==offset and row['end']>offset,'Duplicate or missing chunk range')
        path=root/'predictions'/row['file']
        req(path.is_file() and sha(path)==row['sha256'],f'Prediction chunk missing or checksum mismatch: {path}')
        with np.load(path,allow_pickle=False) as z:
            n=row['end']-row['start']
            req('model_names' in z and list(z['model_names'].astype(str))==sources,'Source ordering mismatch')
            req('classes' in z and list(z['classes'].astype(str))==CLASSES,'Class ordering mismatch')
            req(len(z['image_id'])==n,'Chunk size mismatch')
            if stage=='G02-A':req(str(z['j1_split_sha256'].item())==status['j1_split_sha256'],'J1 split SHA mismatch')
            if stage=='G02-B':req(str(z['n01_manifest_sha256'].item())==status['n01_manifest_sha256'],'N01 manifest mismatch')
            ids.extend(z['image_id'].astype(str));ys.extend(z['y'].astype(int));origin.extend(z['source'].astype(str));group.extend(z['group_id'].astype(str))
            for s in sources:
                a=np.asarray(z['prob__'+s],dtype=float)
                req(a.shape==(n,8) and np.isfinite(a).all() and (a>=0).all() and np.max(abs(a.sum(1)-1))<2e-4,f'{s} bad source probabilities')
                data[s].append(a)
        offset=row['end']
    req(len(ids)==status['images']==status['processed'] and len(ids)==len(set(ids)),'Incomplete or duplicate image predictions')
    req(all(q in ['HAM10000','BCN20000'] for q in origin),'Unexpected source groups')
    req(all(0<=int(q)<8 for q in ys),'Unexpected diagnostic class index')
    return pd.DataFrame({'image_id':ids,'y':ys,'source':origin,'group_id':group}),{s:np.vstack(data[s]) for s in sources}

class Candidates:
    def __init__(self,n):self.n=n;self.p=OrderedDict();self.spec=OrderedDict()
    def add(self,name,family,a,spec):
        req(name not in self.p,f'Duplicate candidate: {name}')
        a=normalized(a);req(a.shape==(self.n,8),f'{name} shape mismatch')
        self.p[name]=a;self.spec[name]={'family':family,**spec}
    def evaluate(self,meta,out,track,scope,boot_names=()):
        out=Path(out);(out/'tables').mkdir(parents=True,exist_ok=True);(out/'predictions').mkdir(exist_ok=True);(out/'reports').mkdir(exist_ok=True)
        rows=[];recs=[];ci=[]
        y=meta.y.to_numpy(dtype=int);orig=meta.source.to_numpy(dtype=str)
        for name,p in self.p.items():
            for source in ['HAM10000','BCN20000']:
                idx=np.flatnonzero(orig==source);yy=y[idx];pp=p[idx]
                if len(idx)==0:continue
                base=subset_score(yy,pp)
                rows.append({'track':track,'scope':scope,'source':source,'method':name,'family':self.spec[name]['family'],'protocol':'full_eight_output',**base})
                narrow=restricted(yy,pp)
                rows.append({'track':track,'scope':scope,'source':source,'method':name,'family':self.spec[name]['family'],'protocol':'restricted_oracle_label_set',**subset_score(yy,narrow)})
                for c in np.unique(yy):
                    pred=pp.argmax(1);q=yy==c
                    recs.append({'source':source,'method':name,'class':CLASSES[c],'support':int(q.sum()),'recall':float((pred[q]==c).mean()),'protocol':'full_eight_output'})
                if name in boot_names:
                    b=group_bootstrap(yy,pp,meta.group_id.to_numpy(dtype=str)[idx],B=int(os.getenv('G03_BOOTSTRAP_N','300')),seed=SEED+int(source=='BCN20000'))
                    ci.append({'source':source,'method':name,**b})
        board=pd.DataFrame(rows).sort_values(['source','protocol','balanced_accuracy'],ascending=[True,True,False])
        board.to_csv(out/'tables/all_ensemble_source_metrics.csv',index=False)
        pd.DataFrame(recs).to_csv(out/'tables/per_class_recall_full_head.csv',index=False)
        pd.DataFrame(ci).to_csv(out/'tables/selected_bootstrap_95ci.csv',index=False)
        ckeys=list(self.p)
        ps=np.stack([self.p[k].astype(np.float32) for k in ckeys],axis=0)
        np.savez_compressed(out/'predictions/ensemble_probabilities.npz',image_id=meta.image_id.astype(str).to_numpy(),
            y=y.astype(np.int32),source=orig,group_id=meta.group_id.astype(str).to_numpy(),
            classes=np.asarray(CLASSES),candidate_names=np.asarray(ckeys),proba=ps)
        jwrite(out/'reports/all_candidate_specs.json',self.spec)
        print('Compared',len(self.p),'methods across HAM/BCN; evaluation rows',len(rows))
        print(board[(board.protocol=='full_eight_output')].groupby('source').head(12)[['source','method','family','balanced_accuracy','macro_f1','unsupported_prediction_rate']].round(4).to_string(index=False))
        return board


## 2. Verify completed model probabilities and cohort

In [2]:
OUT=WORK/'g03b-lesion-aware-all-ensembles-s42'
(OUT/'reports').mkdir(parents=True,exist_ok=True)
SOURCES=['rotation','simclr','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
status_path,status=find_status('g02b_status.json','G02-B')
req(status.get('model_names')==SOURCES,'G02-B source names/order mismatch')
req(status.get('cohort')=='validation',
    'G03-B defaults to N01 validation only. A locked-test result must not be used to choose an ensemble.')
meta,raw=load_chunks(status_path,status,SOURCES,'G02-B')
req(meta.source.value_counts().to_dict().get('HAM10000',0)>0 and meta.source.value_counts().to_dict().get('BCN20000',0)>0,
    'Missing HAM/BCN evaluation cohort')
print('G03-B loaded:',len(meta),'source mix',meta.source.value_counts().to_dict(),
      'N01 manifest sha',status['n01_manifest_sha256'])



G03-B loaded: 2109 source mix {np.str_('HAM10000'): 1060, np.str_('BCN20000'): 1049} N01 manifest sha 04d03deee0f2a01aebf37b48d76bce9d2d17915cfd0d914af0016c0e36432e11


## 3. Evaluate predefined ensemble families without fitting

In [3]:
matches=[]
for path in INPUT.rglob('final_ensemble_config.json'):
    if not path.is_file():continue
    try:
        cfg=get_json(path)
        if cfg.get('stage')=='R07' and cfg.get('sources')==SOURCES and 'all_candidate_specs' in cfg and cfg.get('status')=='COMPLETED':
            matches.append((path,cfg))
    except (ValueError,OSError,TypeError):continue
req(len(matches)==1,f'Attach exactly one completed `lesion-r07-n01-all-ensembles-s42` output with its models. Found: {[str(x[0]) for x in matches]}')
config_path,cfg=matches[0];R07=config_path.parent.parent
req(cfg['classes']==CLASSES and cfg['n01_manifest_sha256']==status['n01_manifest_sha256'],
    'R07 classes/locked N01 manifest do not match G02-B')
req(bool(cfg.get('test_touched')==False) and cfg.get('final_selection')=='validation','Unexpected R07 provenance')
check={k:v for k,v in cfg.items() if k!='frozen_config_sha256'}
expected=hashlib.sha256(json.dumps(check,sort_keys=True,default=str,allow_nan=False).encode()).hexdigest()
req(expected==cfg['frozen_config_sha256'],'R07 frozen configuration SHA mismatch')
orig_status=R07/'reports/final_status.json'
req(orig_status.is_file(),'R07 final status missing')
rs=get_json(orig_status)
req(rs.get('status')=='COMPLETED' and rs.get('frozen_config_sha256')==cfg['frozen_config_sha256'],
    'R07 status and frozen configuration disagree')
print('R07 frozen candidate specification count:',len(cfg['all_candidate_specs']),
      'original validation choice:',cfg['selected_method'])



R07 frozen candidate specification count: 205 original validation choice: ClassBias_Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.4


## 4. Recover and replay original fitted ensembles

In [4]:
cal={s:tscale(raw[s],float(cfg['temperatures'][s])) for s in SOURCES}
provenance=[]
# Compare current G02-B 224px exact preprocessing to R07 saved source predictions at identical IDs.
for s in SOURCES:
    path=R07/'predictions'/f'{s}_meta_val.npz'
    req(path.is_file(),f'Missing R07 source reference: {path}')
    with np.load(path,allow_pickle=True) as z:
        ids=z['val_image_id'].astype(str)
        req(len(set(ids))==len(ids),'R07 validation reference contains duplicate IDs')
        refs=pd.Index(ids).get_indexer(meta.image_id.astype(str).to_numpy())
        req((refs>=0).all(),'G02-B contains IDs absent from R07 held-out validation')
        y_ref=z['val_y'].astype(int)[refs]
        req(np.array_equal(y_ref,meta.y.to_numpy()),f'R07/G02-B {s} target mismatch')
        p=np.asarray(z['val_proba'],dtype=float)[refs]
        delta=np.abs(raw[s]-p)
        provenance.append({'source':s,'mean_abs_probability_difference':float(delta.mean()),
            'p99_abs_probability_difference':float(np.percentile(delta,99)),
            'max_abs_probability_difference':float(delta.max()),
            'status':'CLOSE' if float(delta.mean())<=.004 and float(np.percentile(delta,99))<=.03 else 'REVIEW'})
provenance=pd.DataFrame(provenance)
provenance.to_csv(OUT/'reports/g02b_vs_r07_source_reconstruction.csv',index=False)
print('Source reproduction audit:\n'+provenance.to_string(index=False))
replay_quality='CLOSE' if (provenance.status=='CLOSE').all() else 'REVIEW_REQUIRED'
if replay_quality!='CLOSE':
    print('WARNING: Some source probabilities differ from R07 validation references; review BEFORE publication. No assertion of exact replay.')



Source reproduction audit:
        source  mean_abs_probability_difference  p99_abs_probability_difference  max_abs_probability_difference status
      rotation                         0.000122                        0.001638                        0.009702  CLOSE
        simclr                         0.000076                        0.001240                        0.006845  CLOSE
  FusionConcat                         0.000078                        0.001282                        0.005931  CLOSE
 FusionGatedV2                         0.000086                        0.001521                        0.012091  CLOSE
 FusionGatedV3                         0.000073                        0.001270                        0.007950  CLOSE
ResidualFusion                         0.000073                        0.001351                        0.013217  CLOSE


## 5. Optional sources and robustness checks

In [5]:
specs=cfg['all_candidate_specs'];reg=Candidates(len(meta));stack_cache={}

def replay(name):
    if name in reg.p:return reg.p[name]
    req(name in specs,f'Candidate {name} is not in the frozen R07 configuration')
    spec=specs[name];kind=spec.get('kind')
    if kind=='base':
        s=spec['source'];req(s in SOURCES,'Unknown base source')
        p=cal[s] if spec['calibrated'] else raw[s]
    elif kind in ['arithmetic','geometric','vote']:
        keys=spec['sources'];p=combine(cal,keys,spec['weights'],kind)
    elif kind=='weighted':p=combine(cal,spec['sources'],spec['weights'],'arithmetic')
    elif kind=='gate':p=gated(cal,spec['subset'],spec['measure'],spec['beta'])
    elif kind=='stack':
        source_name=spec['name'];req(source_name==name,'Frozen stacker identity mismatch')
        pth=R07/'models'/f'{source_name}.joblib'
        req(pth.is_file(),f'Original R07 fitted stacker missing: {pth}')
        # joblib is a pickle format. Only load files from your own trusted Kaggle output.
        model=joblib.load(pth)
        p=aligned_predict(model,stacked_features(cal,SOURCES,spec['feature_mode']))
        stack_cache[name]=str(pth)
    elif kind=='blend':
        a=replay(spec['anchor']);b=replay(spec['stack']);alpha=float(spec['alpha'])
        req(0<=alpha<=1,'Invalid frozen blend alpha')
        p=normalized((1-alpha)*a+alpha*b)
    elif kind=='class_bias':p=bias(replay(spec['parent']),spec['offsets'])
    else:raise RuntimeError(f'Unsupported frozen R07 candidate kind={kind!r} in {name}')
    reg.add(name,cfg.get('candidate_families',{}).get(name,kind),p,{'kind':kind,'spec':spec,'fit':'FROZEN_R07'})
    return reg.p[name]

for name in specs:replay(name)
req(len(reg.p)==len(specs),'A frozen R07 candidate was not reproduced')
req(cfg['selected_method'] in reg.p,'R07 validation-selected candidate missing')
req(len(reg.p)==int(rs['candidate_count']),'R07 candidate count differs from final status')



## 6. Results, frozen prediction export and status

In [6]:
# Reproduction is a diagnostic, not an opportunity to refit or optimize.
checks=[]
for name in [cfg['selected_method'],'Base_simclr','Base_ResidualFusion','arithmetic_All6','Stack_extended_balanced']:
    if name not in reg.p:continue
    path=R07/'predictions'/f'candidate_{name}.npy'
    if not path.is_file():continue
    allp=np.load(path,mmap_mode='r',allow_pickle=False)
    idpath=R07/'predictions/final_validation.npz'
    req(idpath.is_file(),'R07 final validation index absent')
    with np.load(idpath,allow_pickle=True) as z:val_ids=z['image_id'].astype(str)
    idx=pd.Index(val_ids).get_indexer(meta.image_id.to_numpy())
    req((idx>=0).all() and allp.shape==(len(val_ids),8),'R07 candidate audit image alignment failed')
    delta=np.abs(reg.p[name]-allp[idx])
    checks.append({'method':name,'mean_abs_probability_difference':float(delta.mean()),
                   'max_abs_probability_difference':float(delta.max())})
pd.DataFrame(checks).to_csv(OUT/'reports/candidate_probability_replay_differences.csv',index=False)

# Preserve the original R07 development leaderboard for side-by-side analysis.
refboard=R07/'tables/all_ensembles_validation.csv'
if refboard.is_file():
    ref=pd.read_csv(refboard)
    req('method' in ref.columns and set(ref.method)==set(reg.p),
        'R07 validation leaderboard candidates differ from frozen replay')
    ref[['method','family','balanced_accuracy','macro_f1']].rename(columns={
        'family':'r07_family','balanced_accuracy':'r07_whole_validation_bacc',
        'macro_f1':'r07_whole_validation_macro_f1'}).to_csv(
        OUT/'reports/r07_original_validation_references.csv',index=False)
    for name,family in zip(ref.method,ref.family):reg.spec[name]['family']=str(family)



## 7. Further checks

In [7]:
selected=cfg['selected_method']
bootstrap=['Base_simclr','Base_ResidualFusion','arithmetic_All6','geometric_All6','vote_All6',
           'Stack_extended_balanced',selected]
board=reg.evaluate(meta,OUT,'lesion_aware',
    'N01_validation_source_stratified_SAME_COHORT_USED_BY_R07_SELECTION',
    boot_names=set(bootstrap)&set(reg.p))
if refboard.is_file():
    compare=board.loc[board.protocol.eq('full_eight_output'),['source','method','balanced_accuracy','macro_f1']].merge(
        pd.read_csv(OUT/'reports/r07_original_validation_references.csv'),on='method',validate='many_to_one')
    compare['difference_vs_whole_validation_bacc']=compare.balanced_accuracy-compare.r07_whole_validation_bacc
    compare.to_csv(OUT/'tables/source_vs_r07_whole_validation.csv',index=False)

jwrite(OUT/'reports/final_status.json',{'stage':'G03-B','status':'COMPLETED','n_images':len(meta),
    'candidate_count':len(reg.p),'g02b_status_sha256':sha(status_path),
    'r07_config_sha256':cfg['frozen_config_sha256'],
    'n01_manifest_sha256':cfg['n01_manifest_sha256'],'original_r07_selected_method':selected,
    'source_reproduction_quality':replay_quality,'reproduction_audit':provenance.to_dict('records'),
    'selection':'NONE: no candidate refit or reselected here',
    'scope':'N01 validation reused by original R07 for candidate selection. Source-specific scores are descriptive, not an independent generalization test.',
    'frozen_stackers_loaded':sorted(stack_cache),'test_touched':False})
print('G03-B complete:',len(reg.p),'frozen R07 candidates; source-replay integrity',replay_quality)


Compared 205 methods across HAM/BCN; evaluation rows 820
  source                                                                   method                    family  balanced_accuracy  macro_f1  unsupported_prediction_rate
BCN20000 ClassBias_Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.4         Thesis class-bias             0.4972    0.4833                          0.0
BCN20000           Blend_Stack_extended_balanced__Weighted_AllConstrained_1__a0.4         Anchored stacking             0.4906    0.4754                          0.0
BCN20000           Blend_Stack_extended_balanced__Weighted_AllConstrained_2__a0.4         Anchored stacking             0.4898    0.4756                          0.0
BCN20000                                                  Weighted_PairwiseGrid_1     Weighted PairwiseGrid             0.4887    0.4837                          0.0
BCN20000                                              Weighted_ResidualDominant_2 Weighted ResidualDominant      